# Data-poisoning a **logistic-regression** victim

The victim performs online mini-batch SGD on the binary cross-entropy loss

$$\ell_i = -\big[y_i\log\sigma(\langle\theta,x_i\rangle) + (1-y_i)\log(1-\sigma(\langle\theta,x_i\rangle))\big],
\qquad \nabla\ell_i = \big(\sigma(\langle\theta,x_i\rangle)-y_i\big)\,x_i$$

with labels $y_i\sim\mathrm{Bernoulli}(\sigma(\langle\theta^\star,x_i\rangle))$, flipped with
probability `noise`.

**Why the method transfers unchanged.** Both tasks have gradient `resid_i * x_i`; only the
residual differs (`<θ,x>-y` vs `σ(<θ,x>)-y`). One SGD step is therefore still

$$\theta_{t+1} = \theta_t - \tfrac{\eta}{m}\sum_i a_i\nabla\ell_i = \theta_t + \sum_i a_i\varphi_i$$

so the transition stays **action-linear** and the whole per-point decomposition
($\alpha$, $B$, all three selection tiers) applies verbatim. Only `Victim.residual`
changes.

**One thing does change: scale.** The logistic residual is bounded in $(-1,1)$ while the
least-squares residual has std $\approx\!2.2$, so $|\varphi|$ is ~5× smaller. Both `eta` and
`c_edit` must be retuned or the attack barely registers — see the sweep in §5.

In [ ]:
import glob
import json
import os
import time
from dataclasses import replace

import numpy as np
import torch
import matplotlib.pyplot as plt
%matplotlib inline

import sys
sys.path.insert(0, "src")

from config import EnvConfig, MethodConfig, TIER_NAME
from env import Environment, is_feasible
from fvi import train_fvi, evaluate, rollout
from agents import NullAgent, RandomAgent, GreedyAgent, MethodAgent
from decomposition import scores_and_interaction

# NB: do not `import viz` / `import diagnostics` -- they call matplotlib.use("Agg")
# at import time, which disables inline plots.
plt.rcParams["figure.dpi"] = 110
plt.rcParams.update({
    "savefig.dpi": 300,
    "font.family": "sans-serif",
    "font.sans-serif": ["DejaVu Sans", "Helvetica", "Arial"],
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "xtick.labelsize": 9.5,
    "ytick.labelsize": 9.5,
    "legend.fontsize": 9.5,
    "legend.frameon": False,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.linewidth": 0.8,
    "axes.grid": True,
    "axes.grid.axis": "y",
    "grid.color": "0.85",
    "grid.linewidth": 0.6,
    "grid.alpha": 0.6,
})
print("torch", torch.__version__)

## Configuration

Retuned for logistic: `eta=1.0` (vs 0.2 for regression) and `c_edit=0.02` (see §8).
Matches `run.py`'s `TASK_DEFAULTS["logistic"]` exactly, so `./run_seeds.sh --task logistic
...` reproduces this notebook's numbers.

§§1-4 below read pre-computed results from `output/` (produced by `run.py --task logistic`
via the terminal) rather than training in this notebook -- same split as
`experiment_linear.ipynb`. §§5-11 still train live in-kernel, since they need a live
trained value net / agent object that a `results.json` summary can't provide (accuracy on
held-out data, the 2-D decision boundary, per-state interaction diagnostics, the
action-linearity violation study, and the rank/duality-gap study that reuses one V̂).

In [ ]:
ECFG = EnvConfig(d=5, m=10, T=10, eta=1.0, noise=0.1, task="logistic")
CFG  = MethodConfig(select_mode="sort", c_edit=0.02, weight_decay=1e-3,
                    K_iters=20, rank_r=2)
EVAL_EPISODES, EVAL_SEED = 200, 999

TRPO_ITERS = 50             # TRPO converges later than the others -- see §§10-11's setup

torch.manual_seed(0)
env = Environment(ECFG, CFG)
print(f"task={ECFG.task}  d={env.d}  m={env.m}  T={env.T}  eta={ECFG.eta}  c_edit={CFG.c_edit}")
z0 = env.stream.sample_batch(8, torch.Generator().manual_seed(0))
print("sample labels (binary):", z0[1].tolist())

## Load results

`run.py --task logistic` trains one selection mode per call (`--mode sort` -> `first-order`,
`--mode lowrank` -> `low-rank`, `--mode exact` -> `exact`), optionally adding `BDQ`/`TRPO`,
and `aggregate.py` folds every seed of one config into a single
`output/{task}_{mode}_{tag}_results_{timestamp}.json`.

`sensitivity/*.sh` (run with `TASK=logistic`) writes plenty of *other* files into this same
`output/` folder, at other configs (different `c_edit`, `m`, ...), so picking "the newest
file" or "whichever file has the most seeds" per mode is not safe on its own -- either
heuristic can silently pair one mode's headline run with another mode's one-off
sensitivity point, badging them as the same comparison. This cell instead fixes an explicit
target config up front (`BASELINE`, reused below by §9), and for each mode only accepts a
file that (a) has no `sens_axis` tag -- i.e. it came from a plain `run_seeds.sh` call, not a
`sensitivity/*.sh` sweep -- and (b) matches `BASELINE` **exactly**.

Two exact matches can still disagree on *what they trained* -- e.g. a `--bdq --trpo` run
and a later plain re-run at the same seed count both match `BASELINE`. Among tied-seed-count
candidates, the one recording more learners wins, with newest path as the final tiebreak. A
mode with no match at all is reported and simply left out of the plots below, rather than
guessed at.

Produce these from the terminal first:
```
./run_seeds.sh --task logistic --seeds 30 --mode sort --bdq --trpo
./run_seeds.sh --task logistic --seeds 30 --mode lowrank
./run_seeds.sh --task logistic --seeds 30 --mode exact
```

In [ ]:
TASK = "logistic"        # matches run.py's --task for this notebook
OUTDIR = "output"

# The exact config this notebook's headline comparison is about -- mirrors run.py's
# TASK_DEFAULTS["logistic"] + the --rank/--k-max argparse defaults. Reused by the
# sensitivity-analysis section (§9) as the point every non-swept axis must match.
# Named ECFG/CFG in the live-training cells below (EnvConfig/MethodConfig objects) --
# this is deliberately a plain dict with a different name so the two never collide.
BASELINE = {"d": 5, "m": 10, "T": 10, "eta": 1.0, "noise": 0.1, "victim_hidden": 8,
            "init_scale": 0.3, "c_edit": 0.02, "weight_decay": 1e-3, "dual_iters": 20,
            "K_iters": 20, "rank_r": 2, "k_max": None}
MODES = ["sort", "lowrank", "exact"]


def cfg_of(run):
    cfg = {**run["env_config"], **run["method_config"]}
    return {f: cfg.get(f) for f in BASELINE}


def matches_baseline(cfg):
    return all(cfg[f] == v for f, v in BASELINE.items())


# Load precisely: for each mode, only accept a file whose recorded config equals BASELINE
# exactly AND whose "sens_axis" tag is unset (i.e. it came from a plain run_seeds.sh call,
# not a sensitivity/*.sh sweep). Both checks matter: BASELINE-matching alone isn't enough
# because rank.sh's own sweep can (and, once you've run it, will) produce lowrank files at
# BASELINE's exact config except rank_r -- exempting rank_r to tolerate that would let a
# sensitivity/*.sh point with more seeds than the real headline run silently win the
# "headline low-rank" slot. Excluding anything with a sens_axis tag up front removes the
# need to exempt anything at all.
pattern = os.path.join(OUTDIR, f"{TASK}_*_results_*.json")
all_paths = glob.glob(pattern)
if not all_paths:
    raise SystemExit(
        f"no aggregated results found matching {pattern!r} -- run e.g.\n"
        f"  ./run_seeds.sh --task logistic --seeds 30 --mode sort --bdq --trpo\n"
        f"first (it calls aggregate.py automatically once all seeds finish)")

files, loaded, ambiguous = {}, {}, {}
for mode in MODES:
    candidates = []
    for path in all_paths:
        data = json.load(open(path))
        if data["mode"] != mode or data.get("sens_axis") is not None or not data["runs"]:
            continue
        if matches_baseline(cfg_of(data["runs"][0])):
            candidates.append((path, data))
    if not candidates:
        continue
    # exact-config matches can still differ in *what they trained*: e.g. one --bdq --trpo
    # run and a later plain re-run of the same seed count both match BASELINE exactly.
    # Prefer, in order: more seeds, then more learners recorded (so a complete run beats a
    # partial one at equal seed count), then newest path as a final tiebreak.
    def candidate_key(pd):
        _, d = pd
        return (len(d["runs"]), len(d["runs"][0]["costs"]), pd[0])
    path, data = max(candidates, key=candidate_key)
    if len(candidates) > 1:
        ambiguous[mode] = [os.path.basename(p) for p, _ in candidates]
    files[mode], loaded[mode] = path, data

if not loaded:
    raise SystemExit(
        f"no file in {OUTDIR}/ matches BASELINE={BASELINE} for any of {MODES} -- run e.g.\n"
        f"  ./run_seeds.sh --task logistic --seeds 30 --mode sort --bdq --trpo\n"
        f"first, or edit BASELINE above to match the run you actually produced.")
missing = [m for m in MODES if m not in loaded]

# merge every mode's per-seed runs into one dict keyed by seed.
merged = {}
for mode, data in loaded.items():
    for run in data["runs"]:
        s = run["seed"]
        dst = merged.setdefault(s, {"costs": {}, "history": {}, "train_times": {},
                                     "decision_times": {},
                                     "env_config": run["env_config"],
                                     "method_config": run["method_config"]})
        for key in ("costs", "history", "train_times", "decision_times"):
            for name, val in run.get(key, {}).items():
                dst[key].setdefault(name, val)

SEEDS_HEADLINE = sorted(merged)
runs = [merged[s] for s in SEEDS_HEADLINE]
ECFG_HEAD = runs[0]["env_config"]
CFG_HEAD = runs[0]["method_config"]

PREFERRED_ORDER = ["null", "random", "greedy", "TRPO", "BDQ", "first-order", "low-rank", "exact"]
present = {n for r in runs for n in r["costs"]}
NAMES = [n for n in PREFERRED_ORDER if n in present] + sorted(present - set(PREFERRED_ORDER))
LEARNERS = [n for n in ["first-order", "low-rank", "exact", "BDQ", "TRPO"] if n in present]

COLORS = {"null": "#5b5b5b", "random": "#9c8f8f", "greedy": "#eb6834", "TRPO": "#4a3aa7",
          "BDQ": "#e34948", "first-order": "#2a78d6", "low-rank": "#1baf7a", "exact": "#12877f"}
MARKERS = {"null": "D", "random": "P", "greedy": "s", "TRPO": "^", "BDQ": "X",
           "first-order": "o", "low-rank": "o", "exact": "o"}
LINESTYLES = {"first-order": "-", "low-rank": "--", "exact": ":"}
_fallback = plt.rcParams["axes.prop_cycle"].by_key()["color"]
for i, n in enumerate(NAMES):
    COLORS.setdefault(n, _fallback[i % len(_fallback)])
    MARKERS.setdefault(n, "o")
    LINESTYLES.setdefault(n, "-")

print("loaded:", {m: os.path.basename(p) for m, p in files.items()})
if missing:
    print(f"no BASELINE-matching file for: {missing} (left out of the plots below)")
if ambiguous:
    for mode, names in ambiguous.items():
        print(f"[{mode}] {len(names)} files match BASELINE -- picked the one with the most "
              f"seeds (ties broken by most learners recorded, then newest): {names}")
print(f"task={TASK}  seeds={SEEDS_HEADLINE}  learners={LEARNERS}  names={NAMES}")
print(f"d={ECFG_HEAD['d']}  m={ECFG_HEAD['m']}  T={ECFG_HEAD['T']}  eta={ECFG_HEAD['eta']}  c_edit={CFG_HEAD['c_edit']}")


## 1. Headline: mean episode cost

Lower is better. Error bars are ±1 std over seeds. All policies are scored on the **same**
held-out episodes; the model-free baselines (TRPO, BDQ) see the identical environment,
constraints and feature set, and every action is asserted feasible.

> **On TRPO's budget.** TRPO trades sample efficiency for stability -- its hard KL bound
> guarantees it cannot collapse, but each step is small, so it converges later.
> `run.py --trpo-iters` defaults to `max(K_iters, 50)` against `K_iters=20` for the others.
> That is **generous to the baseline**, not to us; at matched iterations TRPO is still near
> null and the comparison would flatter our method.

In [ ]:
cost_mean = {n: float(np.mean([r["costs"][n] for r in runs if n in r["costs"]])) for n in NAMES}
cost_std = {n: float(np.std([r["costs"][n] for r in runs if n in r["costs"]])) for n in NAMES}

n = len(NAMES)
fig, ax = plt.subplots(figsize=(max(8, 1.05 * n), 4.6))
ax.bar(range(n), [cost_mean[k] for k in NAMES],
       yerr=[cost_std[k] for k in NAMES], capsize=5, color=[COLORS[k] for k in NAMES])
for i, k in enumerate(NAMES):
    ax.text(i, cost_mean[k] + 0.02 * max(cost_mean.values()), f"{cost_mean[k]:.3f}", ha="center", fontsize=10)
if "null" in cost_mean:
    ax.axhline(cost_mean["null"], color=COLORS["null"], ls="--", lw=1)
ax.set_xticks(range(n))
ax.set_xticklabels(NAMES, rotation=(12 if n <= 6 else 25), fontsize=9,
                    ha=("center" if n <= 6 else "right"))
ax.set_ylabel("mean episode cost (lower = stronger attack)")
ax.set_title(f"Attack strength (d={ECFG_HEAD['d']}, m={ECFG_HEAD['m']}, T={ECFG_HEAD['T']}, "
             f"c_edit={CFG_HEAD['c_edit']}, {len(SEEDS_HEADLINE)} seeds)")
plt.tight_layout()
plt.show()

if "null" in cost_mean:
    nul = cost_mean["null"]
    for k in NAMES:
        if k != "null":
            print(f"{k:12} vs null: {(nul - cost_mean[k]) / nul * 100:5.1f}%")
tiers_present = [t for t in ("first-order", "low-rank", "exact") if t in cost_mean]
if len(tiers_present) > 1:
    base = cost_mean[tiers_present[0]]
    for t in tiers_present[1:]:
        d = base - cost_mean[t]
        print(f"\n{tiers_present[0]} vs {t}: {d:+.3f} ({d / base * 100:+.2f}%)")


## 2. Learning curves — training iterations

`history[name]` is the held-out mean episode cost after every training iteration
(`train_fvi`/`train_bdq`/`train_trpo` called with `return_history=True`, evaluated on the
same fixed `eval_seed` every iteration -- a clean curve). Learners can have different curve
lengths (TRPO's `K_iters` defaults to `max(K_iters, 50)`, longer than FVI's), so each is
plotted against its own iteration count rather than a shared axis.

In [ ]:
def curve_stats(name):
    H = np.array([r["history"][name] for r in runs if name in r["history"]])   # (seeds, K)
    return H.mean(0), H.std(0)

fig, ax = plt.subplots(figsize=(9.5, 5.2))
for name in LEARNERS:
    mu, sd = curve_stats(name)
    it = np.arange(1, len(mu) + 1)
    ax.plot(it, mu, marker=MARKERS[name], linestyle=LINESTYLES[name], color=COLORS[name],
            lw=2, label=name, markevery=max(1, len(it) // 8))
    ax.fill_between(it, mu - sd, mu + sd, color=COLORS[name], alpha=0.12 if len(LEARNERS) > 3 else 0.15)
if "null" in cost_mean:
    ax.axhline(cost_mean["null"], color=COLORS["null"], ls="--", lw=1.5,
               label=f"null ({cost_mean['null']:.2f})")
if "greedy" in cost_mean:
    ax.axhline(cost_mean["greedy"], color=COLORS["greedy"], ls="-.", lw=1.5,
               label=f"greedy ({cost_mean['greedy']:.2f})")
ax.set_xlabel("training iteration")
ax.set_ylabel("held-out mean episode cost")
ax.set_title("Learning curves (shaded = ±1 std over seeds)")
ax.legend(fontsize=9, ncol=(2 if len(LEARNERS) + 2 > 5 else 1))
sec = ax.secondary_xaxis("top", functions=(lambda k: k * CFG_HEAD["N_episodes"] * ECFG_HEAD["T"],
                                            lambda n: n / (CFG_HEAD["N_episodes"] * ECFG_HEAD["T"])))
sec.set_xlabel("environment transitions consumed (approx.: N_episodes × T per iteration)")
plt.show()


## 3. Learning curves — wall-clock

Matched iterations is not matched compute. `train_times[name] / len(history[name])` gives
each learner's mean seconds-per-iteration; re-plotting against cumulative wall-clock
answers *"given a fixed compute budget, which is ahead?"*

In [ ]:
per_it = {n: float(np.mean([r["train_times"][n] / len(r["history"][n])
                             for r in runs if n in r["train_times"] and n in r["history"]]))
          for n in LEARNERS}

fig, ax = plt.subplots(figsize=(9.5, 5.2))
for name in LEARNERS:
    mu, sd = curve_stats(name)
    it = np.arange(1, len(mu) + 1)
    x = it * per_it[name]
    ax.plot(x, mu, marker=MARKERS[name], linestyle=LINESTYLES[name], color=COLORS[name], lw=2,
            label=f"{name} ({per_it[name]:.1f} s/iter)", markevery=max(1, len(it) // 8))
    ax.fill_between(x, mu - sd, mu + sd, color=COLORS[name], alpha=0.12 if len(LEARNERS) > 3 else 0.15)
if "greedy" in cost_mean:
    ax.axhline(cost_mean["greedy"], color=COLORS["greedy"], ls="-.", lw=1.5, label="greedy")
if "null" in cost_mean:
    ax.axhline(cost_mean["null"], color=COLORS["null"], ls="--", lw=1.5, label="null")
ax.set_xscale("log")
ax.set_xlabel("cumulative training wall-clock (s, log scale)")
ax.set_ylabel("held-out mean episode cost")
ax.set_title("Same curves against compute spent")
ax.legend(fontsize=8, ncol=(2 if len(LEARNERS) + 2 > 5 else 1))
plt.show()

for name in LEARNERS:
    mu, _ = curve_stats(name)
    print(f"  {name:12} final={mu[-1]:.3f}  reached in {per_it[name] * len(mu):6.1f}s "
          f"({per_it[name]:.2f} s/iter x {len(mu)} iters)")


## 4. Training time & decision latency

Mean wall-clock to train each learner once (one seed) -- a one-off cost -- and mean
wall-clock per single `agent.act()` call at decision time. Decision latency is recorded for
every evaluated policy, including the untrained baselines (`null`/`random`/`greedy`), since
it's an inference-time cost independent of whether a policy was trained at all.

In [ ]:
train_t = {n: float(np.mean([r["train_times"][n] for r in runs if n in r["train_times"]]))
           for n in LEARNERS}

fig, ax = plt.subplots(figsize=(max(7, 1.3 * len(train_t)), 4.3))
ax.bar(range(len(train_t)), list(train_t.values()), color=[COLORS[k] for k in train_t])
ax.set_xticks(range(len(train_t)))
ax.set_xticklabels(list(train_t), rotation=12, fontsize=9)
for i, v in enumerate(train_t.values()):
    ax.text(i, v + max(train_t.values()) * 0.02, f"{v:.0f}s", ha="center", fontsize=9)
ax.set_ylabel("seconds")
ax.set_title("Mean training time per seed")
plt.tight_layout()
plt.show()

if "first-order" in train_t:
    for n in LEARNERS:
        if n != "first-order":
            print(f"{n}/first-order training time = {train_t[n] / train_t['first-order']:.1f}x")

dec_names = [n for n in NAMES if any(n in r.get("decision_times", {}) for r in runs)]
if not dec_names:
    print("no decision_times recorded in the loaded runs -- re-run with the current "
          "run.py to get this (older results.json files predate it)")
else:
    dec_t = {n: float(np.mean([r["decision_times"][n] for r in runs if n in r.get("decision_times", {})])) * 1000
              for n in dec_names}   # seconds -> ms
    missing_dec = [n for n in NAMES if n not in dec_names]
    if missing_dec:
        print(f"no decision_times for: {missing_dec} (older results.json files predate this field)")

    fig, ax = plt.subplots(figsize=(max(7, 1.3 * len(dec_t)), 4.3))
    ax.bar(range(len(dec_t)), list(dec_t.values()), color=[COLORS[k] for k in dec_t])
    ax.set_xticks(range(len(dec_t)))
    ax.set_xticklabels(list(dec_t), rotation=12, fontsize=9)
    for i, v in enumerate(dec_t.values()):
        ax.text(i, v + max(dec_t.values()) * 0.02, f"{v:.3f}", ha="center", fontsize=9)
    ax.set_ylabel("milliseconds / decision")
    ax.set_title("Mean decision latency (single agent.act() call)")
    plt.tight_layout()
    plt.show()

    if "first-order" in dec_t:
        for n in dec_names:
            if n != "first-order" and dec_t[n] > 0:
                print(f"{n}/first-order decision latency = {dec_t[n] / dec_t['first-order']:.1f}x")


### Train one live model (used by §§5, 7, 11 below)

`results.json` deliberately saves only costs/history/train_times/decision_times -- no model
weights, no environment, no rollouts (see `experiment_linear.ipynb`'s closing note for why).
The sections below need more than that: accuracy on held-out data (§5), per-state
interaction diagnostics (§7), and low-rank's selection quality against a shared V̂ (§11)
all need a live trained value net and live agent objects. This cell retrains one seed at
the same `BASELINE` config as the headline above, purely to supply those objects -- it is
not a substitute for §1's multi-seed numbers, so BDQ/TRPO (already covered there) aren't
retrained here.

In [ ]:
def compare(env, cfg, V, seeds_rng=1, tiers=("sort", "lowrank")):
    """Evaluate every policy on the SAME held-out episodes."""
    agents = {
        "null":   NullAgent(),
        "random": RandomAgent(torch.Generator().manual_seed(2), cfg.balance, cfg.k_max),
        "greedy": GreedyAgent(env.victim, env.theta_target, cfg, mode="sort"),
    }
    for t in tiers:
        agents[TIER_NAME[t]] = MethodAgent(
            V, env.victim, env.theta_target, replace(cfg, select_mode=t),
            torch.Generator().manual_seed(seeds_rng))
    out = {}
    for n, ag in agents.items():
        c = evaluate(env, ag, torch.Generator().manual_seed(EVAL_SEED), EVAL_EPISODES)
        e, g = behaviour(env, ag)
        out[n] = {"cost": c, "edits": e, "gap": g, "agent": ag}
    return out


def behaviour(env, agent, n_episodes=60):
    """Mean edits/step and mean ||theta_t - theta_target||; also asserts feasibility."""
    rng = torch.Generator().manual_seed(EVAL_SEED)
    edits = gap = 0.0
    steps = 0
    for _ in range(n_episodes):
        theta = env.theta_0.clone()
        for t in range(env.T):
            z = env.stream.sample_batch(env.m, rng)
            a = agent.act(theta, z, t)
            assert is_feasible(a, CFG.balance, CFG.k_max), "infeasible action!"
            edits += float(np.abs(a.numpy() - 1).sum())
            theta = env.victim.update(theta, z, a).detach()
            gap += float((theta - env.theta_target).norm())
            steps += 1
    return edits / steps, gap / steps


t0 = time.perf_counter()
V, hist = train_fvi(env, CFG, seed=0, verbose=False, return_history=True,
                    best_checkpoint=True, eval_episodes=EVAL_EPISODES, eval_seed=EVAL_SEED)
print(f"trained in {time.perf_counter()-t0:.0f}s")

res = compare(env, CFG, V)
ORDER = ["null", "random", "greedy", "first-order", "low-rank"]
res = {k: res[k] for k in ORDER if k in res}
nul = res["null"]["cost"]
for name, r in res.items():
    print(f"  {name:12} cost={r['cost']:9.3f}  ({r['cost']/nul:5.3f} x null)"
          f"  edits={r['edits']:5.2f}  gap={r['gap']:7.3f}")
print("\nsee §1 above for the definitive multi-seed headline numbers -- this is one seed's "
      "live model, kept only to supply agents/env/V to §§5, 7, 11.")


## 5. What the attack does to the classifier

Cost is an abstraction; for a classifier the interpretable quantities are **accuracy** and
the **decision boundary**. Accuracy is measured on a large clean test batch drawn from the
true model.

In [ ]:
def accuracy(env, theta, n=4000, seed=7):
    rng = torch.Generator().manual_seed(seed)
    x, y = env.stream.sample_batch(n, rng)
    return float(((torch.sigmoid(x @ theta) > 0.5).float() == y).float().mean())

def final_theta(env, agent, seed=EVAL_SEED, n_ep=40):
    rng = torch.Generator().manual_seed(seed)
    ths = []
    for _ in range(n_ep):
        theta = env.theta_0.clone()
        for t in range(env.T):
            z = env.stream.sample_batch(env.m, rng)
            theta = env.victim.update(theta, z, agent.act(theta, z, t)).detach()
        ths.append(theta)
    return torch.stack(ths).mean(0)

names = list(res)
accs = {n: accuracy(env, final_theta(env, res[n]["agent"])) for n in names}
# accs["θ* (oracle)"] = accuracy(env, env.stream.theta_star)

plt.figure(figsize=(8, 4.2))
ks = list(accs)
_ACC_COLORS = {"null": "#5b5b5b", "random": "#008300", "greedy": "#eb6834",
               "TRPO": "#4a3aa7", "BDQ": "#e34948", "first-order": "#2a78d6",
               "low-rank": "#1baf7a"}
plt.bar(range(len(ks)), [accs[k] for k in ks],
        color=["#e87ba4" if k.startswith("θ*") else _ACC_COLORS.get(k, "#5b5b5b") for k in ks])
# plt.axhline(accs["null"], color="0.5", ls="--", lw=1, label="clean (null) accuracy")
plt.xticks(range(len(ks)), ks, rotation=15, fontsize=8)
plt.ylabel("test accuracy on clean data"); plt.ylim(0, 1)
plt.title("Victim accuracy after the episode (lower = more damage)")
plt.legend(fontsize=9); plt.show()
for k, v in accs.items():
    print(f"  {k:18} accuracy = {v:.3f}")

## 6. Decision boundary per attack policy (d = 2)

A separate 2-D run so the boundary can be drawn, with **every policy on the identical data
stream** — null, random, greedy, TRPO, BDQ, and ours.

> **Why the target is rotated here.** The cost is $\tfrac\lambda2\|\theta-\tilde\theta\|^2$,
> which depends on the *magnitude* of $\theta$ as well as its direction — but a decision
> boundary depends **only on direction**. With the default random $\tilde\theta$ (≈33° from
> $\theta^\star$, and mostly differing in norm) every policy lands on almost the same line
> and the plot shows nothing: measured angles to $\tilde\theta$ were 26°, 23°, 25°, 25°, 26°,
> 28° for the six policies. So for **this visualisation only** we rotate $\tilde\theta$ 90°
> from $\theta^\star$ at a reachable norm, making the attack a rotation the plot can show.
> All quantitative sections use the default random target.

**Reading the panels.** Marker *shape* is the attacker's action (● kept, ✕ deleted,
▲ duplicated) and marker *fill* is the point's **true label** (blue `y=0`, red `y=1`).
Background shading is the victim's predicted $P(y{=}1)$ after the episode, so a point
sitting over the opposite-coloured background is one the victim now **misclassifies**.
Read the black line against the red dashed target: the closer they align, the more
successful the attack. Per-panel counts break the edits down by class, which shows
*which* class each attacker sacrifices.

In [ ]:
INCLUDE_MODEL_FREE_BOUNDARY = True     # BDQ ≈ 3.5 min, TRPO ≈ 1 min; False = skip both

E2 = replace(ECFG, d=2)
C2 = replace(CFG, K_iters=20)
torch.manual_seed(0)
env2 = Environment(E2, C2)

# See the note above: rotate the target so the attack is visible as a rotation.
_R = torch.tensor([[0., -1.], [1., 0.]])
env2.theta_target = (_R @ env2.stream.theta_star) / env2.stream.theta_star.norm() * 1.035

V2 = train_fvi(env2, C2, seed=0, verbose=False, best_checkpoint=True,
               eval_episodes=100, eval_seed=EVAL_SEED)

policies = {
    "null (clean SGD)": NullAgent(),
    "random": RandomAgent(torch.Generator().manual_seed(2), C2.balance, C2.k_max),
    "greedy": GreedyAgent(env2.victim, env2.theta_target, C2, mode="sort"),
}
if INCLUDE_MODEL_FREE_BOUNDARY:
    from bdq import train_bdq, BDQAgent
    from trpo import train_trpo, TRPOAgent
    torch.manual_seed(0)
    Q2 = train_bdq(env2, C2, seed=0, verbose=False, best_checkpoint=True,
                   eval_episodes=100, eval_seed=EVAL_SEED)
    torch.manual_seed(0)
    P2 = train_trpo(env2, replace(C2, K_iters=50), seed=0, verbose=False,
                    best_checkpoint=True, eval_episodes=100, eval_seed=EVAL_SEED)
    policies["TRPO"] = TRPOAgent(P2, env2, C2, torch.Generator().manual_seed(1))
    policies["BDQ"] = BDQAgent(Q2, env2, C2, torch.Generator().manual_seed(1))
policies[TIER_NAME[C2.select_mode]] = MethodAgent(V2, env2.victim, env2.theta_target, C2,
                                        torch.Generator().manual_seed(1))

rng = torch.Generator().manual_seed(3)
batches = [env2.stream.sample_batch(env2.m, rng) for _ in range(env2.T)]   # shared stream


def run_on(agent):
    th = env2.theta_0.clone()
    xs, ys, acts = [], [], []
    for t, z in enumerate(batches):
        a = agent.act(th, z, t)
        xs.append(z[0]); ys.append(z[1]); acts.append(a)
        th = env2.victim.update(th, z, a).detach()
    return th, torch.cat(xs).numpy(), torch.cat(ys).numpy(), torch.cat(acts).numpy()


def angle(u, v):
    c = float(u @ v / (u.norm() * v.norm() + 1e-12))
    return float(np.degrees(np.arccos(np.clip(c, -1, 1))))


star, targ = env2.stream.theta_star, env2.theta_target
gg = torch.linspace(-3, 3, 220)
XX, YY = torch.meshgrid(gg, gg, indexing="ij")
grid = torch.stack([XX.reshape(-1), YY.reshape(-1)], -1)

def draw_boundary(ax, th, color, ls, lw=2.2):
    ax.contour(XX.numpy(), YY.numpy(), (grid @ th).reshape(220, 220).numpy(),
               levels=[0], colors=color, linestyles=ls, linewidths=lw)

ncols = 3
nrows = (len(policies) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 5.1 * nrows),
                         sharex=True, sharey=True, squeeze=False)
CLS = {0.0: "#1f4e9c", 1.0: "#b3202c"}          # class 0 = blue, class 1 = red
rows = []
for ax, (name, ag) in zip(axes.flat, policies.items()):
    th, X, Y, A = run_on(ag)
    for lab in (0.0, 1.0):
        c, sel = CLS[lab], Y == lab
        ax.scatter(X[sel & (A == 1), 0], X[sel & (A == 1), 1], s=20, c=c,
                   alpha=.85, edgecolors="white", linewidths=.5, zorder=3)
        ax.scatter(X[sel & (A == 0), 0], X[sel & (A == 0), 1], s=85, c=c,
                   marker="x", linewidths=2.6, zorder=4)
        ax.scatter(X[sel & (A == 2), 0], X[sel & (A == 2), 1], s=80, c=c,
                   marker="^", edgecolors="black", linewidths=.9, zorder=4)
    draw_boundary(ax, star, "g", ":")
    draw_boundary(ax, targ, "red", "--")
    draw_boundary(ax, th, "k", "-", 2.8)
    acc = accuracy(env2, th)
    d0 = int(((A == 0) & (Y == 0)).sum()); d1 = int(((A == 0) & (Y == 1)).sum())
    u0 = int(((A == 2) & (Y == 0)).sum()); u1 = int(((A == 2) & (Y == 1)).sum())
    rows.append((name, acc, angle(th, targ), d0, d1, u0, u1))
    ax.set_title(f"{name}\nacc={acc:.3f} \n"
                 f"deleted {d0}×y=0 / {d1}×y=1    duplicated {u0}×y=0 / {u1}×y=1",
                 fontsize=9.5)
    ax.set_xlim(-3, 3); ax.set_ylim(-3, 3)
for ax in list(axes.flat)[len(policies):]:
    ax.axis("off")

from matplotlib.lines import Line2D
fig.legend(handles=[
    Line2D([0], [0], color="g", ls=":", lw=2.2, label="θ* boundary (true)"),
    Line2D([0], [0], color="red", ls="--", lw=2.2, label="θ̃ boundary (target)"),
    Line2D([0], [0], color="k", ls="-", lw=2.8, label="victim after episode"),
    Line2D([0], [0], marker="o", color="0.45", ls="None", mec="w", label="kept"),
    Line2D([0], [0], marker="x", color="0.45", ls="None", label="deleted"),
    Line2D([0], [0], marker="^", color="0.45", ls="None", mec="k", label="duplicated"),
    Line2D([0], [0], marker="s", color=CLS[0.0], ls="None", label="true label y=0"),
    Line2D([0], [0], marker="s", color=CLS[1.0], ls="None", label="true label y=1")],
    loc="lower center", ncol=4, fontsize=9.5)
fig.suptitle("Logistic victim (d=2): decision boundary",
             fontsize=13)
fig.tight_layout(rect=(0, 0.09, 1, 0.95)); plt.show()

print(f"{'policy':20}{'acc':>7}{'∠θ̃':>7}{'del y=0':>9}{'del y=1':>9}"
      f"{'dup y=0':>9}{'dup y=1':>9}")
for r in rows:
    print(f"{r[0]:20}{r[1]:>7.3f}{r[2]:>6.0f}°{r[3]:>9d}{r[4]:>9d}{r[5]:>9d}{r[6]:>9d}")
print(f"\nreference: ∠(θ*, θ̃) = {angle(star, targ):.0f}°  "
      f"(null stays near θ*, a successful attack drives the angle to θ̃ toward 0°)")
print("Single episode, single seed, rotated target — illustrative, not a benchmark; "
      "the quantitative comparison is §1.")

## 7. Interaction diagnostics

How much does the quadratic term matter on this task? If `|½aᵀBa| / |aᵀα|` is small, `sort`
is near-optimal and `lowrank` cannot buy much.

In [ ]:
alpha_rel, spec = [], []
rng = torch.Generator().manual_seed(EVAL_SEED)
theta = env.theta_0.clone()
for i in range(60):
    t = i % env.T
    if t == 0:
        theta = env.theta_0.clone()
    z = env.stream.sample_batch(env.m, rng)
    alpha, B = scores_and_interaction(V, theta, z, t, env.victim, env.theta_target,
                                      CFG.lam, need_B=True, use_hessian=CFG.use_hessian_in_B)
    a = res["first-order"]["agent"].act(theta, z, t)
    lin = float(a @ alpha)
    alpha_rel.append(abs(float(0.5 * a @ B @ a)) / (abs(lin) + 1e-12))
    spec.append(np.sort(np.abs(np.linalg.eigvalsh(0.5 * (B + B.T).numpy())))[::-1])
    theta = env.victim.update(theta, z, a).detach()

spec = np.array(spec)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12.5, 4.2))
a1.hist(alpha_rel, bins=25, color="C0", alpha=0.85)
a1.axvline(np.median(alpha_rel), color="k", ls="--", lw=1.5,
           label=f"median = {np.median(alpha_rel):.3f}")
a1.set_xlabel("|½ aᵀBa| / |aᵀα|"); a1.set_ylabel("states")
a1.set_title("Relative interaction (what `sort` discards)"); a1.legend(fontsize=9)

a2.semilogy(np.arange(1, spec.shape[1] + 1), spec.mean(0), "o-", color="C0", lw=2)
a2.set_xlabel("index k"); a2.set_ylabel("mean |eigenvalue| of B")
a2.set_title(f"Spectrum of B  (rank(B) ≤ min(m,d) = {min(ECFG.m, env.d)})")
plt.tight_layout(); plt.show()
print(f"relative interaction: median {np.median(alpha_rel):.4f}  mean {np.mean(alpha_rel):.4f}")

## 8. Why `eta` and `c_edit` had to be retuned

The logistic residual is bounded, so `|φ|` is ~5× smaller than in least squares. At the
regression defaults the attack is invisible; too large an `eta` makes the victim unstable.
This sweep is what set the configuration above.

In [ ]:
rows = []
for eta, ce in [(0.2, 0.1), (0.5, 0.1), (1.0, 0.1), (1.5, 0.1), (2.0, 0.1), (0.2, 0.02), (0.5, 0.02), (1.0, 0.02), (1.5, 0.02), (2.0, 0.02)]:
    c = replace(CFG, c_edit=ce)
    e = replace(ECFG, eta=eta)
    torch.manual_seed(0)
    en = Environment(e, c)
    Vs = train_fvi(en, c, seed=0, verbose=False, best_checkpoint=True,
                   eval_episodes=EVAL_EPISODES, eval_seed=EVAL_SEED)
    ag = MethodAgent(Vs, en.victim, en.theta_target, c, torch.Generator().manual_seed(1))
    gr = GreedyAgent(en.victim, en.theta_target, c, mode="sort")
    n_ = evaluate(en, NullAgent(), torch.Generator().manual_seed(EVAL_SEED), EVAL_EPISODES)
    g_ = evaluate(en, gr, torch.Generator().manual_seed(EVAL_SEED), EVAL_EPISODES)
    m_ = evaluate(en, ag, torch.Generator().manual_seed(EVAL_SEED), EVAL_EPISODES)
    rows.append((eta, ce, n_, g_, m_))
    print(f"eta={eta:<4} c_edit={ce:<5} null={n_:7.3f} greedy={g_:7.3f} "
          f"{TIER_NAME[CFG.select_mode]}={m_:7.3f}  ({m_/n_:.3f} x null)")

r = np.array(rows)
plt.figure(figsize=(7.5, 4.2))
for ce, mk in [(0.1, "o-"), (0.02, "s--")]:
    s = r[r[:, 1] == ce]
    if len(s):
        plt.plot(s[:, 0], s[:, 4] / s[:, 2], mk, lw=2, label=f"{TIER_NAME[CFG.select_mode]}, c_edit={ce}")
plt.axhline(1.0, color="0.7", ls=":", lw=1)
plt.xlabel("victim learning rate eta"); plt.ylabel("cost relative to null")
plt.title("Attack strength vs eta"); plt.legend(fontsize=9); plt.show()

## 9. Sensitivity analysis

Data for this section comes from `sensitivity/*.sh` run with `TASK=logistic` (`c_edit.sh`,
`eta.sh`, `m.sh`, `T.sh`, `d.sh`, `k_max.sh` -- `rank.sh` is excluded by default, same
reasoning as `experiment_linear.ipynb`: `rank(B) <= d` regardless of task, so it needs a
larger `--d` to be informative; §11 below covers rank properly for this task instead).
Each retrains at every value of one axis and writes an
`output/{task}_{mode}_{tag}_results_{timestamp}.json` per value via the usual
`run_seeds.sh` + `aggregate.py` path.

**Recentred for logistic.** The bounded logistic residual makes `|φ|` ~5x smaller than
least squares (see §0), so `c_edit`'s default sweep values (calibrated around regression's
`c_edit=0.1`) are the wrong scale here -- override `VALUES` explicitly, and `BASE_C_EDIT`
for `m.sh`/`k_max.sh` (which move `c_edit` alongside their own axis):

```
TASK=logistic VALUES="0.0 0.005 0.01 0.02 0.04 0.08" ./sensitivity/c_edit.sh --seeds 5
TASK=logistic ./sensitivity/eta.sh --seeds 5                    # default range already brackets 1.0
TASK=logistic BASE_C_EDIT=0.02 ./sensitivity/m.sh --seeds 5
TASK=logistic ./sensitivity/T.sh --seeds 5
TASK=logistic ./sensitivity/d.sh --seeds 5
TASK=logistic BASE_C_EDIT=0.02 ./sensitivity/k_max.sh --seeds 5
```

Membership is read from each file's own `sens_axis` field, not inferred from config
content -- see `experiment_linear.ipynb`'s §5 markdown for why that inference is unsafe.
Each plot has up to three panels (cost, train time, decision latency); a panel with no data
for the current axis is simply omitted. Any axis you haven't run yet is skipped with a
message.

In [ ]:
# field: which env_config/method_config key is this axis's x-value.  mode: which tier's
# output file to read.  "rank" is intentionally absent -- see the markdown above.
AXIS_SPECS = {
    "c_edit": dict(mode="sort",    field="c_edit", xlabel="edit price  c_edit"),
    "eta":    dict(mode="sort",    field="eta",    xlabel="victim learning rate  eta"),
    "m":      dict(mode="sort",    field="m",      xlabel="batch size  m"),
    "T":      dict(mode="sort",    field="T",      xlabel="horizon  T"),
    "d":      dict(mode="sort",    field="d",      xlabel="problem dimension  d"),
    "k_max":  dict(mode="sort",    field="k_max",  xlabel="edit budget  k_max"),
}


def load_sensitivity(axis):
    """{"cost": {value: {learner: [costs]}}, "train_time": {...}, "decision_time": {...}}

    Trusts each file's own "sens_axis" tag (written by run.py --sens-axis, set
    automatically by the matching sensitivity/*.sh script) instead of inferring axis
    membership from config content.

    train_time/decision_time are pooled the same way as cost (one raw value per seed) --
    a name simply won't appear in a metric's points if no matching run recorded it.
    """
    spec = AXIS_SPECS[axis]
    pattern = os.path.join(OUTDIR, f"{TASK}_*_results_*.json")
    metrics = {"cost": {}, "train_time": {}, "decision_time": {}}
    sources = {"cost": "costs", "train_time": "train_times", "decision_time": "decision_times"}
    for path in glob.glob(pattern):
        data = json.load(open(path))
        if data["mode"] != spec["mode"] or data.get("sens_axis") != axis:
            continue
        for run in data["runs"]:
            cfg = {**run["env_config"], **run["method_config"]}
            v = cfg[spec["field"]]
            if v is None:
                continue
            for metric, src in sources.items():
                for name, val in run.get(src, {}).items():
                    metrics[metric].setdefault(v, {}).setdefault(name, []).append(val)
    return metrics


def _errorbar_panel(ax, points, xlabel, ylabel, scale=1.0):
    """One axis's worth of mean+-std error bars, one line per learner present."""
    values = sorted(points)
    names = [n for n in PREFERRED_ORDER if any(n in points[v] for v in values)]
    for name in names:
        xs = [v for v in values if name in points[v]]
        mu = [np.mean(points[v][name]) * scale for v in xs]
        sd = [np.std(points[v][name]) * scale for v in xs]
        ax.errorbar(xs, mu, yerr=sd, marker=MARKERS.get(name, "o"), color=COLORS.get(name),
                    lw=2, capsize=4, label=name)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    return names


def plot_sensitivity(axis):
    spec = AXIS_SPECS[axis]
    metrics = load_sensitivity(axis)
    if not metrics["cost"]:
        print(f"[{axis}] no matching runs in {OUTDIR}/ yet -- run ./sensitivity/{axis}.sh")
        return

    panel_specs = [("cost", "mean episode cost", 1.0),
                   ("train_time", "train time (s)", 1.0),
                   ("decision_time", "decision latency (ms)", 1000.0)]
    panel_specs = [p for p in panel_specs if metrics[p[0]]]

    fig, axes = plt.subplots(1, len(panel_specs), figsize=(6.6 * len(panel_specs), 4.3), squeeze=False)
    legend_handles = None
    for ax, (metric, ylabel, scale) in zip(axes[0], panel_specs):
        _errorbar_panel(ax, metrics[metric], spec["xlabel"], ylabel, scale)
        if legend_handles is None:
            legend_handles = ax.get_legend_handles_labels()
    n_seeds = max(len(next(iter(metrics["cost"][v].values()))) for v in metrics["cost"])
    fig.suptitle(f"Sensitivity: {spec['xlabel']}  (up to {n_seeds} seed(s)/point)")
    axes[0][0].legend(*legend_handles, fontsize=9)
    plt.tight_layout()
    plt.show()


In [ ]:
for axis in AXIS_SPECS:
    plot_sensitivity(axis)


---

**§§10-11 below remain live, in-kernel studies** (unlike §9's CLI-driven sweeps): they use
bespoke victim wrappers (§10) and reuse the shared V̂ trained above (§11), neither of which
`run.py`/`results.json` supports. The next cell defines the shared machinery both need.

In [ ]:
from config import TIER_MODE
from bdq import train_bdq, BDQAgent
from trpo import train_trpo, TRPOAgent
from selection import select_action, lowrank_gap_trace

RUN_VIOLATION = False    # §10  ~1.8 h
RUN_RANK      = True     # §11  seconds (no training)

SEEDS          = (0,)                # single-seed; add more if affordable
K_COMPARE      = CFG.K_iters         # matched budget for ALL learners
SWEEP_LEARNERS = ["first-order", "low-rank", "BDQ", "TRPO"]   # retrained per point
SWEEP_FREE     = ["null", "greedy"]                           # training-free
COLORS = {"null": "#5b5b5b", "greedy": "#eb6834", "TRPO": "#4a3aa7",
          "BDQ": "#e34948", "first-order": "#2a78d6", "low-rank": "#1baf7a",
          "exact": "#e87ba4"}
# ours (any selector tier) shares one marker; every baseline gets its own
MARKERS = {"null": "D", "greedy": "s", "TRPO": "^", "BDQ": "X",
           "first-order": "o", "low-rank": "o", "exact": "o"}


def sweep_behaviour(env, agent, n_episodes=40, seed=EVAL_SEED):
    """(edits per step, all-feasible?)."""
    rng = torch.Generator().manual_seed(seed)
    edits, steps, ok = 0.0, 0, True
    for _ in range(n_episodes):
        theta = env.theta_0.clone()
        for t in range(env.T):
            z = env.stream.sample_batch(env.m, rng)
            a = agent.act(theta, z, t)
            ok &= is_feasible(a, CFG.balance, CFG.k_max)
            edits += float(np.abs(a.numpy() - 1).sum())
            theta = env.victim.update(theta, z, a).detach()
            steps += 1
    return edits / steps, ok


def decision_latency_ms(env, agent, n=200, seed=1):
    """Mean wall-clock of one agent.act() call, in milliseconds."""
    rng = torch.Generator().manual_seed(seed)
    theta, z = env.theta_0.clone(), env.stream.sample_batch(env.m, rng)
    agent.act(theta, z, 0)                                    # warm-up
    t0 = time.perf_counter()
    for k in range(n):
        agent.act(theta, z, k % env.T)
    return (time.perf_counter() - t0) / n * 1e3


def build_agents(env, cfg, seed, learners=SWEEP_LEARNERS):
    """Train `learners` on this env/cfg; return {name: agent} incl. the free baselines."""
    ags = {"null": NullAgent(),
           "greedy": GreedyAgent(env.victim, env.theta_target,
                                 replace(cfg, select_mode="sort"), mode="sort")}
    for name in learners:
        torch.manual_seed(seed)                       # identical init across learners
        if name == "BDQ":
            net = train_bdq(env, cfg, seed=seed, verbose=False, best_checkpoint=True,
                            eval_episodes=EVAL_EPISODES, eval_seed=EVAL_SEED)
            ags[name] = BDQAgent(net, env, cfg, torch.Generator().manual_seed(seed + 1))
        elif name == "TRPO":
            cfg_t = replace(cfg, K_iters=max(cfg.K_iters, TRPO_ITERS))
            net = train_trpo(env, cfg_t, seed=seed, verbose=False, best_checkpoint=True,
                             eval_episodes=EVAL_EPISODES, eval_seed=EVAL_SEED)
            ags[name] = TRPOAgent(net, env, cfg_t, torch.Generator().manual_seed(seed + 1),
                                  deterministic=True)
        else:
            cfg_t = replace(cfg, select_mode=TIER_MODE[name])
            V_s = train_fvi(env, cfg_t, seed=seed, verbose=False, best_checkpoint=True,
                            eval_episodes=EVAL_EPISODES, eval_seed=EVAL_SEED)
            ags[name] = MethodAgent(V_s, env.victim, env.theta_target, cfg_t,
                                    torch.Generator().manual_seed(seed + 1))
    return ags


def sweep(values, make_cfgs, seeds=(0,), learners=SWEEP_LEARNERS, label="x",
          post_env=None, n_beh=40):
    """1-D sweep: retrain every learner at every point, score on the shared episodes.

    make_cfgs(v) -> (EnvConfig, MethodConfig) for axis value v.
    post_env(env, v) may mutate the freshly built Environment (used by §10).
    Returns {name: {"cost", "cost_sd", "edits"}}, each a list over `values`.
    """
    names = SWEEP_FREE + list(learners)
    res = {n: {"cost": [], "cost_sd": [], "edits": []} for n in names}
    for v in values:
        ecfg_v, cfg_v = make_cfgs(v)
        per_c = {n: [] for n in names}
        per_e = {n: [] for n in names}
        for s in seeds:
            torch.manual_seed(s)
            env_v = Environment(ecfg_v, cfg_v)
            if post_env is not None:
                post_env(env_v, v)
            ags = build_agents(env_v, cfg_v, s, learners)
            for n in names:
                per_c[n].append(evaluate(env_v, ags[n],
                                torch.Generator().manual_seed(EVAL_SEED), EVAL_EPISODES))
                e_, ok = sweep_behaviour(env_v, ags[n], n_episodes=n_beh)
                assert ok, f"{n} emitted an infeasible action at {label}={v}"
                per_e[n].append(e_)
        for n in names:
            res[n]["cost"].append(float(np.mean(per_c[n])))
            res[n]["cost_sd"].append(float(np.std(per_c[n])))
            res[n]["edits"].append(float(np.mean(per_e[n])))
        print(f"  {label}={v}: " +
              "  ".join(f"{n}={res[n]['cost'][-1]:.2f}" for n in names), flush=True)
    return res


print("shared machinery for §§10-11 ready -- flags:", {"10": RUN_VIOLATION, "11": RUN_RANK})


## 10. Graded violation of action-linearity — the scope of the method

The advantage is bought with a modelling assumption, `θ_{t+1} = θ_t + Σ aᵢ φᵢ`. §0 argues
that assumption survives the move from least squares to logistic *unchanged*, because both
have gradient `resid_i · x_i` and only the residual differs. That is a claim about the
**exact** case; this section asks how much mismatch the advantage tolerates.

Two knobs, each attacking a different half of *"sum-structured loss + exactly one step"*:

| knob | what it breaks | exact at |
|---|---|---|
| **global grad-norm clip `τ`** | sum structure — the clip applies to `Σ aᵢ∇ℓᵢ` *after* aggregation, so the update is no longer separable | `τ = ∞` |
| **inner SGD steps `n`** | "exactly one step" — steps 2…`n` re-linearise at a `θ` the attacker never modelled | `n = 1` |

Per-sample clipping is deliberately not used: it is still exactly affine in `a`, so it
violates nothing. It is the *aggregate* clip — what `torch.nn.utils.clip_grad_norm_` does —
that breaks separability.

**Logistic-specific note.** The residual is already bounded in `(-1,1)`, so the raw update
norm is small and less variable than in the regression case. Expect the clip to bite later
(at smaller `ρ`) but the multi-step knob to bite *harder*: `σ` saturation means re-linearising
at a new `θ` can change the gradient direction, not just its length.

**The wrapper overrides `update` (the true dynamics) but leaves `phi_batch` (the attacker's
model) linear.** Every learner trains on the true dynamics; only our `α`/`B` carry the
assumption — BDQ and TRPO assumed nothing and are the control.

**How to read it.** A clipped or multi-step victim is a genuinely *different control
problem*: `null` moves too (hence the `cost/null` y-axis) and the baselines are **not**
guaranteed flat. Read the **gap between our curve and the best baseline at each violation
level**, not our own degradation against the exact case.

In [ ]:
class ClipVictim:
    """Clips the AGGREGATE update to norm tau -- breaks separability in `a`.

    phi_batch (the attacker's model) is left linear on purpose: the mismatch between
    `update` and `theta + a @ Phi` IS the experiment.
    """

    def __init__(self, base, tau):
        self.base, self.tau = base, tau
        self.eta, self.task = base.eta, getattr(base, "task", "logistic")

    def residual(self, theta, z):
        return self.base.residual(theta, z)

    def phi_batch(self, theta, z):
        return self.base.phi_batch(theta, z)

    def update(self, theta, z, a):
        step = a @ self.base.phi_batch(theta, z)
        nrm = step.norm()
        if self.tau is not None and float(nrm) > self.tau:
            step = step * (self.tau / nrm)
        return theta + step


class MultiStepVictim:
    """n SGD steps on the same reweighted batch -- breaks 'exactly one step'.

    n=1 is exactly the base victim, so the sweep has a clean exact anchor.
    """

    def __init__(self, base, n_steps):
        self.base, self.n_steps = base, n_steps
        self.eta, self.task = base.eta, getattr(base, "task", "logistic")

    def residual(self, theta, z):
        return self.base.residual(theta, z)

    def phi_batch(self, theta, z):
        return self.base.phi_batch(theta, z)     # attacker still assumes ONE step

    def update(self, theta, z, a):
        th = theta
        for _ in range(self.n_steps):
            th = th + a @ self.base.phi_batch(th, z)   # re-linearised at the new theta
        return th


def linearity_error(env, agent, n_states=60, seed=EVAL_SEED):
    """Mean ‖update(θ,z,a) − (θ + Σ aᵢφᵢ)‖ / ‖Σ aᵢφᵢ‖ over states the policy visits."""
    rng = torch.Generator().manual_seed(seed)
    errs, theta = [], env.theta_0.clone()
    for i in range(n_states):
        if i % env.T == 0:
            theta = env.theta_0.clone()
        z = env.stream.sample_batch(env.m, rng)
        a = agent.act(theta, z, i % env.T)
        pred = theta + a @ env.victim.phi_batch(theta, z)
        true = env.victim.update(theta, z, a).detach()
        errs.append(float((true - pred).norm() / ((pred - theta).norm() + 1e-12)))
        theta = true
    return float(np.mean(errs))


RHOS    = [None, 1.0, 0.6, 0.4, 0.25, 0.12]   # clip threshold / typical step norm
NSTEPS  = [1, 2, 3, 4]

if RUN_VIOLATION:
    # reference step norm: median ‖Σ aᵢφᵢ‖ under the unclipped greedy policy
    torch.manual_seed(SEEDS[0])
    _env0 = Environment(ECFG, replace(CFG, K_iters=K_COMPARE))
    _g0 = GreedyAgent(_env0.victim, _env0.theta_target, CFG, mode="sort")
    _rng, _norms, _th = torch.Generator().manual_seed(EVAL_SEED), [], _env0.theta_0.clone()
    for i in range(200):
        if i % _env0.T == 0:
            _th = _env0.theta_0.clone()
        _z = _env0.stream.sample_batch(_env0.m, _rng)
        _a = _g0.act(_th, _z, i % _env0.T)
        _norms.append(float((_a @ _env0.victim.phi_batch(_th, _z)).norm()))
        _th = _env0.victim.update(_th, _z, _a).detach()
    STEP0 = float(np.median(_norms))
    print(f"typical unclipped step norm = {STEP0:.4f}  -> tau = rho * {STEP0:.4f}\n")

    print("[aggregate gradient clipping]")
    res_clip = sweep(RHOS,
                     lambda r: (ECFG, replace(CFG, K_iters=K_COMPARE)),
                     seeds=(SEEDS[0],), label="rho",
                     post_env=lambda env, r: setattr(
                         env, "victim", ClipVictim(env.victim,
                                                   None if r is None else r * STEP0)))

    print("[multiple inner SGD steps]")
    res_step = sweep(NSTEPS,
                     lambda n: (ECFG, replace(CFG, K_iters=K_COMPARE)),
                     seeds=(SEEDS[0],), label="n_steps",
                     post_env=lambda env, n: setattr(
                         env, "victim", MultiStepVictim(env.victim, n)))

    # measured violation at each point (uses the greedy policy: no training needed)
    def violation_curve(values, wrap):
        out = []
        for v in values:
            torch.manual_seed(SEEDS[0])
            e = Environment(ECFG, replace(CFG, K_iters=K_COMPARE))
            e.victim = wrap(e.victim, v)
            out.append(linearity_error(
                e, GreedyAgent(e.victim, e.theta_target, CFG, mode="sort")) * 100)
        return out

    vio_clip = violation_curve(RHOS, lambda vic, r: ClipVictim(vic, None if r is None
                                                               else r * STEP0))
    vio_step = violation_curve(NSTEPS, MultiStepVictim)

    fig, (c1, c2) = plt.subplots(1, 2, figsize=(12.5, 4.3), sharey=True)
    for ax, xs, res_, xlab, ttl, inv in [
            (c1, range(len(RHOS)), res_clip,
             "clip threshold  τ / typical step", "Aggregate gradient clipping", True),
            (c2, NSTEPS, res_step,
             "inner SGD steps per round", "Multiple inner steps", False)]:
        nul = np.array(res_["null"]["cost"])
        for n in res_:
            ax.plot(list(xs), np.array(res_[n]["cost"]) / nul, f"{MARKERS[n]}-",
                    color=COLORS[n], lw=2, label=n)
        ax.axhline(1.0, color="0.7", ls=":", lw=1)
        ax.set_xlabel(xlab); ax.set_title(ttl); ax.legend(fontsize=8)
        if inv:
            ax.set_xticks(list(xs))
            ax.set_xticklabels(["exact" if r is None else f"{r:g}" for r in RHOS])
    c1.set_ylabel("cost / null")
    fig.suptitle("How much action-linearity violation does the advantage survive?", y=1.02)
    plt.tight_layout(); plt.show()

    print(f"\n{'setting':>22}{'violation':>12}{'first-order':>14}{'BDQ':>10}{'TRPO':>10}")
    for lbl, res_, vio, vals in [("clip rho", res_clip, vio_clip, RHOS),
                                 ("inner steps", res_step, vio_step, NSTEPS)]:
        nul = np.array(res_["null"]["cost"])
        for i, v in enumerate(vals):
            tag = f"{lbl}={'exact' if v is None else v}"
            print(f"{tag:>22}{vio[i]:>11.1f}%"
                  f"{res_['first-order']['cost'][i]/nul[i]:>14.3f}"
                  f"{res_['BDQ']['cost'][i]/nul[i]:>10.3f}"
                  f"{res_['TRPO']['cost'][i]/nul[i]:>10.3f}")
    print("\nquote the violation column, not rho: it is the transferable number.")
else:
    print("skipped — set RUN_VIOLATION = True")

## 11. Rank `r` — how much of the interaction `B` do you need?

`rank(B) <= min(m, d)` always (`decomposition.py` builds it as `Phi @ (.) @ Phi.T`), and on
the linear notebook that ceiling combined with a spectrum dominated ~3-4x by the top
eigendirection made `r=1..5` pick nearly-identical actions -- flat enough that `rank` was
dropped from that notebook's default sensitivity sweep entirely (see
`sensitivity/rank.sh`'s header). **On logistic the answer may differ**: §7 measures the
relative interaction `|½aᵀBa| / |aᵀα|` on this task, and the same rank cap still applies, so
sweeping `r` up to it settles whether the interaction is real here or just as negligible.

**This sweep is free.** It reuses the `V̂` trained in the live-model cell above and varies
the selector at decision time only, so cost differences are *pure selection quality* on an
identical value function. `exact` (BQP) is the attainable floor; the duality gap is
`low-rank`'s own certificate.

In [ ]:
if RUN_RANK:
    env_r, V_r, cfg_r = env, V, CFG          # the V̂ trained in §1
    RANKS = list(range(1, min(ECFG.m, env_r.d) + 1))

    def score_selector(cfg_sel):
        ag = MethodAgent(V_r, env_r.victim, env_r.theta_target, cfg_sel,
                         torch.Generator().manual_seed(1))
        c = evaluate(env_r, ag, torch.Generator().manual_seed(EVAL_SEED), EVAL_EPISODES)
        return c, decision_latency_ms(env_r, ag)

    c_sort, l_sort = score_selector(replace(cfg_r, select_mode="sort"))
    c_exact, l_exact = score_selector(replace(cfg_r, select_mode="exact"))
    costs_r, lat_r = [], []
    for r in RANKS:
        c, l = score_selector(replace(cfg_r, select_mode="lowrank", rank_r=r))
        costs_r.append(c); lat_r.append(l)
        print(f"  r={r}: cost={c:.4f}  latency={l:.3f} ms", flush=True)

    # low-rank's own optimality certificate at each rank
    rng_r = torch.Generator().manual_seed(EVAL_SEED)
    gaps, theta_r = {r: [] for r in RANKS}, env_r.theta_0.clone()
    for i in range(60):
        if i % env_r.T == 0:
            theta_r = env_r.theta_0.clone()
        z = env_r.stream.sample_batch(env_r.m, rng_r)
        alpha, B = scores_and_interaction(V_r, theta_r, z, i % env_r.T, env_r.victim,
                                          env_r.theta_target, cfg_r.lam, need_B=True,
                                          use_hessian=cfg_r.use_hessian_in_B)
        for r in RANKS:
            gaps[r].append(lowrank_gap_trace(alpha, B, c_edit=cfg_r.c_edit,
                                             k_max=cfg_r.k_max, balance=cfg_r.balance,
                                             r=r, dual_iters=cfg_r.dual_iters)[-1])
        a_r = select_action(alpha, None, "sort", c_edit=cfg_r.c_edit,
                            balance=cfg_r.balance, k_max=cfg_r.k_max)
        theta_r = env_r.victim.update(theta_r, z, a_r).detach()

    fig, (q1, q2, q3) = plt.subplots(1, 3, figsize=(16, 4.3))
    q1.plot(RANKS, costs_r, "o-", color=COLORS["low-rank"], lw=2, label="low-rank(r)")
    q1.axhline(c_sort, color=COLORS["first-order"], ls="--", lw=2,
               label=f"first-order ({c_sort:.3f})")
    q1.axhline(c_exact, color=COLORS["exact"], ls="-.", lw=2, label=f"exact ({c_exact:.3f})")
    q1.set_xlabel("rank r"); q1.set_ylabel("mean episode cost")
    q1.set_title(f"Selection quality (shared V̂, rank(B) ≤ {min(ECFG.m, env_r.d)})")
    q1.set_xticks(RANKS); q1.legend(fontsize=8)

    q2.plot(RANKS, lat_r, "o-", color=COLORS["low-rank"], lw=2, label="low-rank(r)")
    q2.axhline(l_sort, color=COLORS["first-order"], ls="--", lw=2, label="first-order")
    q2.axhline(l_exact, color=COLORS["exact"], ls="-.", lw=2, label="exact")
    q2.set_xlabel("rank r"); q2.set_ylabel("ms per action"); q2.set_yscale("log")
    q2.set_title("Price of the rank"); q2.set_xticks(RANKS); q2.legend(fontsize=8)

    mg = [float(np.mean(gaps[r])) for r in RANKS]
    neg = [int(np.sum(np.array(gaps[r]) < 0)) for r in RANKS]
    q3.plot(RANKS, mg, "o-", color=COLORS["low-rank"], lw=2)
    q3.axhline(0, color="k", lw=0.8)
    q3.set_xlabel("rank r"); q3.set_ylabel("mean final duality gap")
    q3.set_title("Certificate quality (negative = bound invalid)")
    q3.set_xticks(RANKS)
    plt.tight_layout(); plt.show()

    print(f"\nfirst-order {c_sort:.4f}   exact {c_exact:.4f}   "
          f"headroom {(c_sort-c_exact)/c_sort*100:+.2f}%")
    print(f"best low-rank: r={RANKS[int(np.argmin(costs_r))]} "
          f"({min(costs_r):.4f}, {(c_sort-min(costs_r))/c_sort*100:+.2f}% vs first-order)")
    print("negative gaps by rank:", {r: f"{n}/60" for r, n in zip(RANKS, neg)})
else:
    print("skipped — set RUN_RANK = True")

## Takeaways

1. **The method transfers with a one-line change** (`Victim.residual`) because
   action-linearity is a property of *per-point SGD*, not of least squares.
2. **Scale must be retuned.** Bounded logistic residuals shrink `|φ|` ~5×, which by the
   same `c_edit/|α|` economics makes a fixed edit price ~5× more punitive.
3. **Damage is interpretable** for a classifier: accuracy drops and the decision boundary
   visibly rotates toward `θ̃`.
4. **Margins are smaller than in regression.** The method still wins, but by less. The
   obvious explanation — bounded residuals compressing the `α` spectrum — was tested and
   **does not hold** (dispersion 1.29 vs 1.40, essentially equal), so the cause is still
   open. Candidates: σ-saturation capping every attacker, or a harder value-learning
   problem. Worth resolving before publishing a logistic headline number.